In [1]:
import numpy as np

In [ ]:
class SGDRegressor:
    """
    Constructor for SGDRegressor
    
    Parameters:
    learning_rate: the step size used in each update.
    epochs: No of passes over the training dataset
    batch_size: Number of samples to be used in each batch.
    reg (str): Type of regularization ('l1' or 'l2'); None if no regularization.
    reg_param (float): Regularization parameter.
    
    The weights and bias are initialized as None and will be set during the fit method.
    """
    def __init__(self, learning_rate=0.01, epochs=50, batch_size=1, reg=None, reg_param=0.0):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.reg = reg
        self.reg_param = reg_param
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        """Fit the SGDRegressor to the training data"""
        # Convert to numpy arrays (safe for pandas DataFrames from Kaggle)
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)  # ensure 1D

        m, n = X.shape
        self.weights = np.zeros(n)
        self.bias = 0.0

        for _ in range(self.epochs):
            indices = np.random.permutation(m)
            X_shuffled = X[indices]
            y_shuffled = y[indices]

            for i in range(0, m, self.batch_size):
                X_batch = X_shuffled[i:i + self.batch_size]
                y_batch = y_shuffled[i:i + self.batch_size]

                # Forward pass
                pred = np.dot(X_batch, self.weights) + self.bias

                # Error = pred - y  (standard convention for MSE gradient)
                error = pred - y_batch

                # Gradients for Mean Squared Error (including the factor of 2)
                gradient_w = (2 / self.batch_size) * np.dot(X_batch.T, error)
                gradient_b = (2 / self.batch_size) * np.sum(error)

                if self.reg == 'l2':
                    gradient_w += 2 * self.reg_param * self.weights
                elif self.reg == 'l1':
                    gradient_w += self.reg_param * np.sign(self.weights)

                # SGD update (this is the exact rule from our earlier lesson!)
                self.weights -= self.learning_rate * gradient_w
                self.bias -= self.learning_rate * gradient_b

        return self   

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.dot(X, self.weights) + self.bias

    def compute_loss(self, X, y):
        """Returns RMSE"""
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)
        pred = self.predict(X)
        mse = np.mean((y - pred) ** 2)
        reg_loss = 0.0
        if self.reg == 'l1':
            reg_loss = self.reg_param * np.sum(np.abs(self.weights))
        elif self.reg == 'l2':
            reg_loss = self.reg_param * np.sum(self.weights ** 2)
        return np.sqrt(mse + reg_loss)  

    def get_weights(self):
        return self.weights